__STAR with Competition for RNA-Polymerase Simulations__

Alfred

In [8]:
# Libraries for mathematical analysis
import numpy as np
from skimage import measure
import sympy as sym
import pylab as pl
import matplotlib.pyplot as plt
import scipy.integrate
import biocircuits

# Libraries to visualize results
import bokeh.io
from bokeh.io import show
from bokeh.plotting import figure
from bokeh.plotting import column
import bokeh.palettes
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.models import Range1d
from bokeh.io import export_svgs
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from bokeh.layouts import row

bokeh.io.output_notebook()

Loading BokehJS ...

<u>STAR Model with Competition for RNA Polymerase</u>

![STAR Competition Reaction Scheme](STAR_Competition_Reaction.png)

The system is governed by the following equations:
\begin{align*}
&r^{tot} = r + \sum_{i=1}^{n} r_i\\
&\dot{r_3} = k_3^+ d_3 r - k_3^- r_3 - a r_3 m + d c - \delta r_3\\
&\dot{r_1} =  k_1^+ d_1 r - k_1^- r_1 - \alpha_1 r_1 - \delta r_1\\
&\dot{r_2} = k_2^+ d_2 r - k_2^- r_2 - \alpha_2 r_2 - \delta r_2\\
&\dot{m} = \alpha_1 r_1 - \phi m - a r_3 m + dc - bzm + uq\\
&\dot{z} = \alpha_2 r_2 - \phi z - bzm + uq\\
&\dot{c} = a r_3 m - dc - vc\\
&\dot{q} = bzm - uq - \phi q\\
&\dot{y} = vc - \delta y\\
\end{align*}

Where:
- $D_1 = \text{Plasmid for STAR}$
- $D_2 = \text{Plasmid for Sequestering RNA}$
- $D_3 = \text{Plasmid for Target RNA}$
- $R = \text{RNA Polymerase}$
- $R_1 = \text{Polymerase bound to DNA for Input RNA}$
- $R_2 = \text{Polymerase Bound to DNA for Sequestering RNA}$
- $R_3 = \text{Polymerase Bound to DNA for Target RNA}$
- $M = \text{STAR}$
- $C = \text{STAR-Target RNA Complex}$
- $Y = \text{Output mRNA}$
- $Z = \text{Sequestering RNA}$
- $Q = \text{Sequestered STAR}$

<u>Simulated Decision Boundaries & Activation Functions</u>

In [9]:
# Function to do the 2D projection of decision boundaries with normalization  <-- From the EBCP LATAM 2025 Workshop
def contourf(p, x, y, z, title=None, palette="Spectral11", normal = True):
    """Make a filled contour plot given x, y, z data given in 2D arrays."""

    # Normalize the z values
    if normal:
      z_min = z.min()
      z_max = z.max()
      z_normalized = (z - z_min) / (z_max - z_min)  # Normalize to range [0, 1]
    else:
      z_normalized = z

    # Add zero padding at the boundaries for better visualization
    N = z_normalized.shape[1]
    z0 = np.c_[z_normalized, np.zeros(N)]
    z0[-1, -1] = 1.  # Ensure the padding contains a value within range

    # Plot the normalized values
    p.image(
        image=[z0],
        x=x.min(),
        y=y.min(),
        dw=(x.max() - x.min()) * (1 + 1 / N),
        dh=x.max() - x.min(),
        palette=palette,
        alpha=0.8,
    )

    # Color mapping based on the normalized z0 values
    color = LinearColorMapper(palette=palette, low=z0.min(), high=z0.max())
    cb = ColorBar(color_mapper=color, location=(0, 0), width=10)
    p.add_layout(cb, 'right')

    return ()

In [10]:
#Function to help calculate the ODEs
def starCompODEs(X, t, rtot, d1, d2, d3, rates):

    r3, r1, r2, m, z, c, q, y = X

    #Unpack rates from array
    k3p = rates["k_3+"] #Binding of polymerase to X3
    k3m = rates["k_3-"] #Unbinding of polymerase to X3
    k1p = rates["k_1+"] #Binding of polymerase to X1
    k1m = rates["k_1-"] #Unbinding of polymerase to X1
    k2p = rates["k_2+"] #Binding of polymerase to X2
    k2m = rates["k_2-"] #Unbinding of polymerase to X2
    alpha3 = rates["alpha3"] #Release of transript
    alpha1 = rates["alpha1"] # RNA production from R1
    alpha2 = rates["alpha2"] # RNA production from R2
    phi = rates["phi"] #Decay of RNA
    delta = rates["delta"] #Decay of output RNA
    a = rates["a"] #Binding of target RNA & STAR
    d = rates["d"] #Dissociation of target RNA & STAR
    b = rates["b"] #Binding of STAR & sequesterer
    u = rates["u"] #Dissociation of STAR & sequesterer

    #By mass conservation
    r = rtot - r3 - r2 - r1 #This allows us to avoid writing out the ODE for r

    #Write out the ODEs
    dr3 = k3p*d3*r - k3m*r3 - a*r3*m - delta*r3
    dr1 = k1p*d1*r - k1m*r1 - alpha1*r1 - delta*r1
    dr2 = k2p*d2*r - k2m*r2 - alpha2*r2 - delta*r2
    dm = alpha1*r1 - phi*m - a*r3*m + d*c - b*z*m + u*q
    dz = alpha2*r2 - phi*z - b*z*m + u*q
    dc = a*r3*m - d*c - alpha3*c
    dq = b*z*m - u*q - phi*q
    dy = alpha3*c - delta*y

    return [dr3, dr1, dr2, dm, dz, dc, dq, dy]

In [33]:
# Define a grid of input values (d1, d2)
d1 = np.linspace(0, 1, 50)
d2 = np.linspace(0, 1, 50)
D1, D2 = np.meshgrid(d1, d2)

#Activation function inputs
d1Vals = np.linspace(0, 10, 50) #Not sure why we need to go up to 10 to see meaningful graphs
d2Vals = [5, 3, 1]

#D3 is currently just a scalar --> Simulations are not varying its input
D3 = 1
rtot = 1

# Simulation time
t = np.linspace(0, 200, 400)

#Initial species states
X0 = [0, #r3
      0, #r1
      0, #r2
      0, #m
      0, #z
      0, #c
      0, #q
      0] #y

#Kinetics parameters
ratesDefault = {
    'k_3+': np.array([10, 10, 10]),
    'k_3-': np.array([1, 1, 1]),
    'k_1+': np.array([10, 10, 10]),
    'k_1-': np.array([1, 1, 1]),
    'k_2+': np.array([10, 10, 10]),
    'k_2-': np.array([1, 1, 1]),
    'alpha3': np.array([1, 1, 1]),
    'alpha1': np.array([1, 1, 1]),
    'alpha2': np.array([1, 1, 1]),
    'phi': np.array([0.1, 0.1, 0.1]),
    'delta': np.array([1, 1, 1]),
    'a': np.array([10, 10, 10]),
    'd': np.array([0.1, 1, 10]),
    'b': np.array([100, 100, 100]),
    'u': np.array([0.2, 0.2, 0.2])
}

In [34]:
#getVariantRates() --> Extracts variant-specific rates from ratesDefault
def getVariantRates(ratesArray, variantIndex):
    rates = {}
    for key, arr in ratesArray.items():
        rates[key] = arr[variantIndex]
    return rates

numVariants = len(ratesDefault['phi']) #phi is in all models --> Best choice

#Bokeh color palette for heatmaps + activation functions
colors = bokeh.palettes.OrRd3

#Array to store all results
results = []

#Range through each variant --> Solve ODEs + Plot heatmap + Plot activation function --> Append to Results
for variantIndex in range(numVariants):
    #Get rates for this current iteration's variant
    rates = getVariantRates(ratesDefault, variantIndex)

    #ODE SOLUTIONS
    #Array for storing scipy.integrate solutions
    gridSolutions = [[None for _ in range(len(d1))] for _ in range(len(d2))]
    
    # Initialize a matrix to store the steady-state output y
    Y = np.zeros_like(D1)

    # Iterate over the grid values for all variable input D1 and D2 --> Solve the ODEs
    for i in range(len(d1)):
        for j in range(len(d2)):
            solutions = scipy.integrate.odeint(starCompODEs, X0, t, args=(rtot, D1[j, i], D2[j, i], D3, rates))
            gridSolutions[j][i] = solutions
            Y[j, i] = solutions[-1, 7] #Extract the steady state output y

    #HEATMAPS
    p_heatmap = figure(width=350, height=300, title=f"Variant {variantIndex}")

    #Make the heatmap
    contourf(
        p_heatmap,
        D1,
        D2,
        Y,
        palette=bokeh.palettes.Oranges8[::-1],
        normal=True
    )

    p_heatmap.xaxis.axis_label = "D1"
    p_heatmap.yaxis.axis_label = "D2"

    #ACTIVATION FUNCTIONS
    activationFunctions = {}
    p_activation = figure(width=350, height=300, title=f"Variant {variantIndex}")

    for d2j, color in zip(d2Vals, colors):

        curves = []
        output_y = np.zeros_like(d1Vals)

        for i, d1i in enumerate(d1Vals):

            sol = scipy.integrate.odeint(
                starCompODEs, X0, t,
                args=(rtot, d1i, d2j, D3, rates)
            )

            curves.append(sol)
            output_y[i] = sol[-1, 7]

        activationFunctions[d2j] = curves

        p_activation.line(
            d1Vals,
            output_y,
            line_width=3,
            color=color,
            legend_label=f"D2 = {d2j}"
        )

    p_activation.xaxis.axis_label = "D1"
    p_activation.yaxis.axis_label = "y"
    p_activation.legend.location = "top_left"

    #Append ODE solutions, heatmaps, and activation functions for all variants into the results array
    results.append({
        "variantIndex": variantIndex,
        "rates": rates,
        "solutions": {
            "grid": gridSolutions,
            "activation": activationFunctions
        },
        "heatmaps": p_heatmap,
        "activationFunctions": p_activation
    })

In [35]:
#Plot results
for res in results:
    print(f"\nVariant {res['variantIndex']}")
    for k, v in res["rates"].items():
        print(f"{k}: {v}")

    show(row(res["heatmaps"], res["activationFunctions"]))


Variant 0
k_3+: 10
k_3-: 1
k_1+: 10
k_1-: 1
k_2+: 10
k_2-: 1
alpha3: 1
alpha1: 1
alpha2: 1
phi: 0.1
delta: 1
a: 10
d: 0.1
b: 100
u: 0.2



Variant 1
k_3+: 10
k_3-: 1
k_1+: 10
k_1-: 1
k_2+: 10
k_2-: 1
alpha3: 1
alpha1: 1
alpha2: 1
phi: 0.1
delta: 1
a: 10
d: 1.0
b: 100
u: 0.2



Variant 2
k_3+: 10
k_3-: 1
k_1+: 10
k_1-: 1
k_2+: 10
k_2-: 1
alpha3: 1
alpha1: 1
alpha2: 1
phi: 0.1
delta: 1
a: 10
d: 10.0
b: 100
u: 0.2


In [14]:
#3D-Plot of the activation functions
fig = go.Figure(data=[
    go.Surface(z=Y, x=D1, y=D2, colorscale="Viridis")
])

fig.update_layout(
    scene=dict(
        xaxis_title='STAR (D1)',
        yaxis_title='Sequesterer RNA (D2)',
        zaxis_title='Output (Y)'
    ),
    width=800,
    height=600
)

fig.show()